<a href="https://colab.research.google.com/github/myrondza10/visa_sme_growth_copilot/blob/main/SME_Copilot_VISA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
SME Growth Co-Pilot — Visa Data + AI Hackathon (UAE) Prototype
================================================================
Features in one app:
  1) AI Marketing Co-Pilot: turns raw "Visa-style" transaction data into
     customer segments (RFM) + ready-to-send campaign copy, plus cohort
     contact-list management and one-click email send-out.
  2) Look-Alike Customer Finder & Large Network Graph: multi-tier merchant
     ecosystem spend graph for cross-promotional partnerships.
  3) AI ROI Engine: predicts joint campaign performance, incremental revenue,
     acquisition cost, net ROI, and forward cashflow for both participating
     merchants.
  4) Strategy & Expansion: location demand/competition analytics and
     merchant benchmarking against category peers.
  5) Raw Transaction Data: tucked away at the bottom of the page for anyone
     who wants to view/filter the underlying transaction feed.
"""

import math
import random
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime, timedelta
import gradio as gr
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

random.seed(42)
np.random.seed(42)

# ----------------------------------------------------------------------
# 1. SYNTHETIC DATA GENERATION & EXPANDED MERCHANT ECOSYSTEM
# ----------------------------------------------------------------------

SME_PROFILES = {
    "Café (Jumeirah)": {
        "category": "Food & Beverage",
        "avg_ticket": 45,
        "n_customers": 180,
    },
    "Boutique Gym (Al Barsha)": {
        "category": "Fitness & Wellness",
        "avg_ticket": 250,
        "n_customers": 90,
    },
    "Hair & Beauty Salon (Marina)": {
        "category": "Personal Care",
        "avg_ticket": 180,
        "n_customers": 140,
    },
    "Independent Retail Store (JLT)": {
        "category": "Retail - Fashion",
        "avg_ticket": 220,
        "n_customers": 200,
    },
}

CATEGORY_AFFINITY = {
    "Food & Beverage": {
        "Fitness & Wellness": 0.42,
        "Personal Care": 0.31,
        "Retail - Fashion": 0.24,
        "Grocery": 0.55,
        "Entertainment": 0.38,
        "Specialty Coffee": 0.68,
        "Organic Markets": 0.49,
    },
    "Fitness & Wellness": {
        "Food & Beverage": 0.47,
        "Personal Care": 0.39,
        "Retail - Fashion": 0.28,
        "Health & Pharmacy": 0.44,
        "Sportswear": 0.61,
        "Nutrition & Supplements": 0.58,
        "Spa & Wellness": 0.52,
    },
    "Personal Care": {
        "Food & Beverage": 0.33,
        "Retail - Fashion": 0.45,
        "Fitness & Wellness": 0.30,
        "Beauty & Cosmetics": 0.58,
        "Spa & Wellness": 0.50,
        "Luxury Apparel": 0.37,
    },
    "Retail - Fashion": {
        "Food & Beverage": 0.29,
        "Personal Care": 0.41,
        "Fitness & Wellness": 0.22,
        "Accessories": 0.53,
        "Footwear": 0.47,
        "Jewelry & Watches": 0.36,
        "Fine Dining": 0.34,
    },
}

EXPANDED_MERCHANT_NAMES = {
    "Fitness & Wellness": [
        "FlexFit Studio",
        "PureCore Gym",
        "Zen Pilates Room",
        "Apex Crossfit",
        "Pulse Cycling Hub",
    ],
    "Food & Beverage": [
        "Bean & Batter Café",
        "Levantine Table",
        "Nomad Coffee Roasters",
        "Artisan Bakery JLT",
        "Bistro Coastal",
    ],
    "Personal Care": [
        "Glow Beauty Bar",
        "The Grooming Lounge",
        "Silk Salon",
        "Urban Barber Co.",
        "Nail Lounge & Spa",
    ],
    "Retail - Fashion": [
        "Loom Concept Store",
        "Atelier 9",
        "Sandy Lane Boutique",
        "Moda Thread Works",
        "Urban Edge Wear",
    ],
    "Grocery": ["Fresh Corner Market", "Organic Food Store", "Green Grocery Bay"],
    "Entertainment": ["CineLounge Marina", "VR World UAE", "Golf Simulator Club"],
    "Health & Pharmacy": ["VitalCare Pharmacy", "Wellness First Pharmacy"],
    "Sportswear": ["Rally Sports Co.", "Motion Athletics UAE"],
    "Beauty & Cosmetics": ["Lumière Cosmetics", "Velvet Rose Beauty"],
    "Spa & Wellness": ["Serenity Spa House", "Hammams & Care Marina"],
    "Accessories": ["Charm & Co.", "Leather & Craft Studio"],
    "Footwear": ["StepOut Shoes", "Sole Culture UAE"],
    "Specialty Coffee": ["Espresso Lab", "Roast & Grind Jumeirah"],
    "Organic Markets": ["Bio Orchard Dubai", "Farm-to-Table Hub"],
    "Nutrition & Supplements": ["Peak Performance Fuel", "NutriLife Store"],
    "Luxury Apparel": ["Maison de Elegance", "Velvet Couture"],
    "Jewelry & Watches": ["Crown & Gem JLT", "Aurum Jewelers"],
    "Fine Dining": ["Le Grand Table", "Azure Coast Dining"],
}

CATEGORY_COLORS = {
    "Food & Beverage": "#f59e0b",
    "Fitness & Wellness": "#10b981",
    "Personal Care": "#ec4899",
    "Retail - Fashion": "#8b5cf6",
    "Grocery": "#84cc16",
    "Entertainment": "#06b6d4",
    "Health & Pharmacy": "#14b8a6",
    "Sportswear": "#3b82f6",
    "Beauty & Cosmetics": "#f43f5e",
    "Spa & Wellness": "#a855f7",
    "Accessories": "#eab308",
    "Footwear": "#6366f1",
    "Specialty Coffee": "#d97706",
    "Organic Markets": "#65a30d",
    "Nutrition & Supplements": "#059669",
    "Luxury Apparel": "#c026d3",
    "Jewelry & Watches": "#ca8a04",
    "Fine Dining": "#e11d48",
}


def generate_transactions(sme_name: str) -> pd.DataFrame:
    profile = SME_PROFILES[sme_name]
    n = profile["n_customers"]
    today = datetime.today()

    rows = []
    for cust_id in range(1, n + 1):
        tier = np.random.choice(
            ["champion", "regular", "lapsing", "new"], p=[0.15, 0.40, 0.30, 0.15]
        )

        if tier == "champion":
            n_txn = np.random.randint(8, 15)
            last_gap = np.random.randint(0, 10)
        elif tier == "regular":
            n_txn = np.random.randint(4, 8)
            last_gap = np.random.randint(5, 30)
        elif tier == "lapsing":
            n_txn = np.random.randint(2, 5)
            last_gap = np.random.randint(45, 120)
        else:
            n_txn = np.random.randint(1, 2)
            last_gap = np.random.randint(0, 15)

        last_date = today - timedelta(days=int(last_gap))
        for i in range(n_txn):
            txn_date = last_date - timedelta(
                days=int(np.random.exponential(scale=12)) * i
            )
            amount = max(
                10,
                np.random.normal(
                    profile["avg_ticket"], profile["avg_ticket"] * 0.3
                ),
            )
            rows.append({
                "transaction_id": f"TXN-{random.randint(100000, 999999)}",
                "customer_id": f"C{cust_id:04d}",
                "merchant_category": profile["category"],
                "date": txn_date.strftime("%Y-%m-%d %H:%M"),
                "amount_aed": round(amount, 2),
                "payment_method": np.random.choice(
                    ["Visa Contactless", "Visa Digital Wallet", "Visa Chip & PIN"],
                    p=[0.60, 0.30, 0.10],
                ),
            })

    df = pd.DataFrame(rows)
    df["date_dt"] = pd.to_datetime(df["date"])
    df = df.sort_values("date_dt", ascending=False).drop(columns=["date_dt"])
    return df


def compute_rfm(df: pd.DataFrame) -> pd.DataFrame:
    today = datetime.today()
    df_calc = df.copy()
    df_calc["date_dt"] = pd.to_datetime(df_calc["date"])

    grouped = (
        df_calc.groupby("customer_id")
        .agg(
            last_purchase=("date_dt", "max"),
            frequency=("date_dt", "count"),
            monetary=("amount_aed", "sum"),
        )
        .reset_index()
    )
    grouped["recency_days"] = (today - grouped["last_purchase"]).dt.days

    def segment(row):
        if row["recency_days"] > 60:
            return "Win-Back (Lapsing)"
        if row["recency_days"] <= 15 and row["frequency"] >= 6:
            return "Champions"
        if row["frequency"] <= 2 and row["recency_days"] <= 20:
            return "New Customers"
        return "Regulars"

    grouped["segment"] = grouped.apply(segment, axis=1)
    return grouped


# ----------------------------------------------------------------------
# 1b. CUSTOMER DIRECTORY (name + email per cohort)
# ----------------------------------------------------------------------
# In production, replace this with a real CRM / Visa-linked customer
# directory lookup (name + consented email address per customer_id).
# For this demo we deterministically generate a synthetic name + a
# safe @example.com email per customer_id, so re-running the app for the
# same SME always yields the same "contact list" per cohort.

FIRST_NAMES = [
    "Sara", "Ahmed", "Fatima", "James", "Priya", "Omar", "Layla", "Noah",
    "Mei", "Yusuf", "Aisha", "Daniel", "Hana", "Karim", "Zainab", "Liam",
]
LAST_NAMES = [
    "Khan", "Ali", "Smith", "Patel", "Hassan", "Brown", "Chen", "Ibrahim",
    "Kumar", "Nasser", "Rahman", "Lopez",
]


def get_customer_directory(sme_name: str) -> dict:
    """Returns {customer_id: {"name": ..., "email": ...}} for an SME.

    Deterministic per (sme_name, customer_id) so the same customer always
    maps to the same contact details across reruns/tabs.
    """
    profile = SME_PROFILES[sme_name]
    n = profile["n_customers"]
    directory = {}
    for cust_id in range(1, n + 1):
        cid = f"C{cust_id:04d}"
        rnd = random.Random(f"{sme_name}::{cid}")
        first = rnd.choice(FIRST_NAMES)
        last = rnd.choice(LAST_NAMES)
        name = f"{first} {last}"
        email = f"{first.lower()}.{last.lower()}{cust_id}@example.com"
        directory[cid] = {"name": name, "email": email}
    return directory


# User-supplied contact lists, keyed by (sme_name, segment). When a list has
# been saved for a given SME + cohort, it's used instead of the synthetic
# demo directory. In-memory only (resets if the app restarts) — swap for a
# real database/CRM table in production.
COHORT_CONTACTS: dict = {}


def parse_contact_list_text(text: str) -> list:
    """Parses freeform 'Name, email@example.com' lines (one per row) into a
    list of {"name": ..., "email": ...} dicts. Tolerant of a header row and
    of email/name being in either order."""
    contacts = []
    if not text:
        return contacts
    for raw_line in text.strip().splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.lower().startswith("name") and "email" in line.lower():
            continue  # skip an optional header row like "Name, Email"
        parts = [p.strip() for p in line.split(",")]
        name, email = None, None
        if len(parts) >= 2 and "@" in parts[1]:
            name, email = parts[0], parts[1]
        elif len(parts) >= 2 and "@" in parts[0]:
            email, name = parts[0], parts[1]
        elif len(parts) == 1 and "@" in parts[0]:
            email = parts[0]
            name = parts[0].split("@")[0]
        if name and email and "@" in email:
            contacts.append({"name": name, "email": email})
    return contacts


def save_cohort_contacts(sme_name: str, segment: str, contacts_text: str) -> str:
    contacts = parse_contact_list_text(contacts_text)
    if not contacts:
        return (
            "⚠️ No valid contacts found. Enter one person per line as "
            "`Name, email@example.com`."
        )
    COHORT_CONTACTS[(sme_name, segment)] = contacts
    return (
        f"✅ Saved **{len(contacts)}** contact(s) to the **{segment}** cohort for "
        f"**{sme_name}**. Campaigns/joint offers sent to this cohort will now use "
        "this list instead of the synthetic demo directory."
    )


def clear_cohort_contacts(sme_name: str, segment: str) -> str:
    existed = COHORT_CONTACTS.pop((sme_name, segment), None) is not None
    if existed:
        return (
            f"🗑️ Cleared the custom contact list for **{segment}** ({sme_name}). "
            "Will fall back to the synthetic demo directory."
        )
    return f"No custom contact list was saved for **{segment}** ({sme_name})."


def preview_cohort_contacts(sme_name: str, segment: str) -> pd.DataFrame:
    contacts = COHORT_CONTACTS.get((sme_name, segment), [])
    if not contacts:
        return pd.DataFrame({
            "Message": [
                f"No custom contact list saved yet for '{segment}' — sends will "
                "use the synthetic demo directory until you save one above."
            ]
        })
    return pd.DataFrame(contacts)


def get_segment_customers(sme_name: str, segment: str) -> pd.DataFrame:
    """Returns the cohort (segment) member list with name + email attached.

    Uses a saved custom contact list for this (sme_name, segment) if present;
    otherwise falls back to the synthetic RFM-derived demo directory.
    """
    custom_contacts = COHORT_CONTACTS.get((sme_name, segment))
    if custom_contacts:
        rows = [
            {
                "customer_id": f"CUSTOM-{i + 1}",
                "name": c["name"],
                "email": c["email"],
                "recency_days": None,
                "frequency": None,
                "monetary": None,
                "segment": segment,
            }
            for i, c in enumerate(custom_contacts)
        ]
        return pd.DataFrame(rows)

    directory = get_customer_directory(sme_name)
    df = generate_transactions(sme_name)
    rfm = compute_rfm(df)
    seg_df = rfm[rfm["segment"] == segment].copy()
    seg_df["name"] = seg_df["customer_id"].map(
        lambda c: directory.get(c, {}).get("name", "Customer")
    )
    seg_df["email"] = seg_df["customer_id"].map(
        lambda c: directory.get(c, {}).get("email", "")
    )
    return seg_df[
        ["customer_id", "name", "email", "recency_days", "frequency", "monetary", "segment"]
    ].sort_values("recency_days")


CAMPAIGN_TEMPLATES = {
    "Win-Back (Lapsing)": {
        "insight": (
            "{pct}% of your customers haven't returned in 60+ days. This is"
            " your biggest revenue-recovery opportunity."
        ),
        "whatsapp": (
            "Hey {name}! 👋 We miss you at {sme}. Here's 20% off your next visit"
            " this week only — just show this message. Come say hi! 🙌"
        ),
        "sms": (
            "{sme}: We miss you! Enjoy 20% off your next visit, valid 7 days."
            " Reply STOP to opt out."
        ),
        "email_subject": "We haven't seen you in a while, {name} 💛",
        "email_body": (
            "Hi {name},\n\nIt's been a bit since your last visit to {sme}. As a"
            " valued customer, here's an exclusive 20% off your next purchase —"
            " valid for the next 7 days.\n\nWe'd love to have you"
            " back!\n\nWarmly,\n{sme} Team"
        ),
    },
    "Champions": {
        "insight": (
            "{pct}% of your customers are Champions — high frequency, recent"
            " visits. Reward and retain them before a competitor does."
        ),
        "whatsapp": (
            "Hi {name}! 🌟 You're one of our top customers at {sme}. As a"
            " thank-you, enjoy early access to our new offer + a free add-on"
            " this week!"
        ),
        "sms": (
            "{sme}: You're a VIP! Enjoy a free add-on on your next visit this"
            " week. Thank you for your loyalty."
        ),
        "email_subject": "You're one of our VIPs, {name} 🌟",
        "email_body": (
            "Hi {name},\n\nYou're one of our most loyal customers at {sme}, and"
            " we want to say thank you. Enjoy a complimentary add-on on your"
            " next visit, plus early access to upcoming offers.\n\nSee you"
            " soon!\n{sme} Team"
        ),
    },
    "New Customers": {
        "insight": (
            "{pct}% are New Customers on their first or second visit — the"
            " critical window to convert them into regulars."
        ),
        "whatsapp": (
            "Welcome to {sme}! 🎉 So glad you visited. Book your next visit"
            " within 2 weeks and get 15% off — let's make this a habit!"
        ),
        "sms": (
            "{sme}: Thanks for visiting! Book again within 2 weeks for 15% off"
            " your next purchase."
        ),
        "email_subject": "Welcome to {sme} — here's 15% for your next visit",
        "email_body": (
            "Hi {name},\n\nThank you for choosing {sme}! We'd love to see you"
            " again. Book your next visit within 2 weeks and enjoy 15% off.\n\nSee"
            " you soon,\n{sme} Team"
        ),
    },
    "Regulars": {
        "insight": (
            "{pct}% are steady Regulars. Small nudges (bundles, referral"
            " incentives) can move them toward Champion status."
        ),
        "whatsapp": (
            "Hi {name}! Thanks for being a regular at {sme} 💙 Refer a friend"
            " this month and you'll both get 10% off!"
        ),
        "sms": (
            "{sme}: Refer a friend this month and you both get 10% off. Thanks"
            " for being a regular!"
        ),
        "email_subject": "A little thank-you from {sme}",
        "email_body": (
            "Hi {name},\n\nWe appreciate you being a regular at {sme}. This"
            " month, refer a friend and you'll both receive 10% off your next"
            " visit.\n\nThanks for your continued support,\n{sme} Team"
        ),
    },
}

SAMPLE_NAMES = ["Sara", "Ahmed", "Fatima", "James", "Priya", "Omar"]


def generate_copy(segment: str, sme_name: str, pct: float) -> dict:
    """Renders the campaign templates, resolving {sme}/{pct} now but keeping
    {name} as a literal placeholder so each recipient can be personalized
    individually at send time."""
    t = CAMPAIGN_TEMPLATES[segment]
    sme_short = sme_name.split(" (")[0]
    pct_val = round(pct, 1)

    def render(s: str) -> str:
        # Temporarily escape {name} so .format() doesn't choke on it, then
        # restore the literal placeholder afterwards.
        return (
            s.replace("{name}", "\0NAME\0")
            .format(sme=sme_short, pct=pct_val)
            .replace("\0NAME\0", "{name}")
        )

    return {
        "insight": t["insight"].format(pct=pct_val),
        "whatsapp": render(t["whatsapp"]),
        "sms": render(t["sms"]),
        "email_subject": render(t["email_subject"]),
        "email_body": render(t["email_body"]),
    }


def generate_joint_offer(
    sme_name: str, partner_name: str, partner_category: str, overlap_pct: int
) -> dict:
    sme_short = sme_name.split(" (")[0]
    discount = 15 if overlap_pct < 40 else 20

    insight = (
        f"{overlap_pct}% of {sme_short}'s customers also spend at businesses like"
        f" **{partner_name}** ({partner_category}). A joint offer could reach a"
        " highly qualified shared audience without paid ads."
    )

    # {{name}} below renders as a literal "{name}" placeholder, personalized
    # per-recipient at send time — matches generate_copy()'s behavior.
    whatsapp = (
        f"Hey {{name}}! 🎉 {sme_short} has teamed up with {partner_name} this"
        f" month. Show your receipt from either place and get {discount}% off at"
        " the other. Two of your favorite spots, one offer! 💛"
    )

    sms = (
        f"{sme_short} x {partner_name}: Show a receipt from either and get"
        f" {discount}% off at the other, this month only. Enjoy!"
    )

    email_subject = (
        f"{sme_short} + {partner_name} — a joint offer just for you 🎁"
    )
    email_body = (
        f"Hi {{name}},\n\nWe've partnered with {partner_name} this month! Since a"
        " lot of our customers already love both places, we wanted to make it"
        f" official.\n\nShow a receipt from {sme_short} at {partner_name} (or"
        f" vice versa) and get {discount}% off — valid for the next 30"
        f" days.\n\nEnjoy exploring both!\n{sme_short} & {partner_name}"
    )

    return {
        "insight": insight,
        "whatsapp": whatsapp,
        "sms": sms,
        "email_subject": email_subject,
        "email_body": email_body,
    }


def run_copilot(sme_name: str, focus_segment: str):
    df = generate_transactions(sme_name)
    rfm = compute_rfm(df)

    counts = rfm["segment"].value_counts()
    total = len(rfm)
    seg_summary = pd.DataFrame({
        "Segment": counts.index,
        "Customers": counts.values,
        "% of Base": [round(100 * v / total, 1) for v in counts.values],
    }).sort_values("Customers", ascending=False)

    pct = seg_summary.loc[seg_summary["Segment"] == focus_segment, "% of Base"]
    pct_val = float(pct.values[0]) if len(pct) else 0.0

    copy = generate_copy(focus_segment, sme_name, pct_val)

    insight_text = f"### 💡 Insight\n{copy['insight']}"
    whatsapp = copy["whatsapp"]
    sms = copy["sms"]
    email = f"Subject: {copy['email_subject']}\n\n{copy['email_body']}"

    return seg_summary, insight_text, whatsapp, sms, email


# ----------------------------------------------------------------------
# 1c. EMAIL SENDING — pulls name/email per cohort and sends the drafted copy
# ----------------------------------------------------------------------

def preview_segment_recipients(sme_name: str, focus_segment: str, max_recipients: int) -> pd.DataFrame:
    """Shows who would actually receive the campaign for this cohort."""
    seg_df = get_segment_customers(sme_name, focus_segment)
    cols = ["customer_id", "name", "email", "recency_days", "frequency", "monetary"]
    return seg_df[cols].head(int(max_recipients))


def send_campaign_emails(
    sme_name: str,
    focus_segment: str,
    email_draft: str,
    smtp_host: str,
    smtp_port: float,
    sender_email: str,
    app_password: str,
    dry_run: bool,
    max_recipients: float,
) -> str:
    """Sends (or dry-run previews) the email_draft ("Subject: ...\\n\\nBody")
    to every customer in the chosen cohort, personalizing {name} per
    recipient using the SME's customer directory."""

    seg_df = get_segment_customers(sme_name, focus_segment)
    if seg_df.empty:
        return f"⚠️ No customers currently in the **{focus_segment}** cohort for this SME."

    seg_df = seg_df.head(int(max_recipients))

    try:
        subject_line, body_template = email_draft.split("\n\n", 1)
        subject_template = subject_line.replace("Subject:", "").strip()
    except ValueError:
        return "⚠️ Couldn't parse the email draft — generate a campaign first."

    if not dry_run:
        if not sender_email or not app_password:
            return "⚠️ Enter a sender email and app password, or enable Dry Run."

    server = None
    if not dry_run:
        try:
            server = smtplib.SMTP_SSL(smtp_host, int(smtp_port))
            server.login(sender_email, app_password)
        except Exception as e:
            return f"❌ SMTP connection/login failed: {e}"

    logs, sent, failed = [], 0, 0
    for _, row in seg_df.iterrows():
        name, email = row["name"], row["email"]
        if not email:
            continue

        subject = subject_template.format(name=name) if "{name}" in subject_template else subject_template
        body = body_template.format(name=name) if "{name}" in body_template else body_template

        if dry_run:
            logs.append(f"📝 DRY RUN → **{name}** <{email}> — Subject: {subject}")
            sent += 1
            continue

        try:
            msg = MIMEMultipart("alternative")
            msg["Subject"] = subject
            msg["From"] = sender_email
            msg["To"] = email
            msg.attach(MIMEText(body, "plain", "utf-8"))
            server.sendmail(sender_email, email, msg.as_string())
            logs.append(f"✅ Sent → **{name}** <{email}>")
            sent += 1
        except Exception as e:
            logs.append(f"❌ Failed → {email}: {e}")
            failed += 1

    if server is not None:
        server.quit()

    mode = "DRY RUN (no emails actually sent)" if dry_run else "LIVE SEND"
    header = (
        f"### 📤 Campaign Send Report — {focus_segment}\n"
        f"**Mode:** {mode}  \n**Sent:** {sent}  **Failed:** {failed}  "
        f"**Cohort size (capped):** {len(seg_df)}\n\n---\n"
    )
    return header + "\n".join(logs[:100])


# ----------------------------------------------------------------------
# 2. LARGE ECOSYSTEM NETWORK GRAPH GENERATOR
# ----------------------------------------------------------------------

def build_large_network_graph(sme_name: str, total_nodes: int = 25) -> go.Figure:
    """
    Builds a dynamic, multi-concentric large graph network representing
    the local merchant ecosystem connected by anonymized Visa spend affinity.
    """
    sme_short = sme_name.split(" (")[0]
    sme_cat = SME_PROFILES[sme_name]["category"]

    affinities = CATEGORY_AFFINITY.get(sme_cat, {})
    all_cats = list(EXPANDED_MERCHANT_NAMES.keys())

    nodes = []
    # Center Node (Level 0)
    nodes.append({
        "id": sme_short,
        "label": sme_short,
        "category": sme_cat,
        "level": 0,
        "size": 48,
        "color": "#1d4ed8",  # Visa Blue
        "overlap": 100,
        "x": 0.0,
        "y": 0.0,
    })

    # Ring 1: Direct High-Affinity Partners (Level 1)
    num_ring1 = min(6, total_nodes - 1)
    ring1_cats = list(affinities.keys())[:num_ring1]

    for cat in all_cats:
        if len(ring1_cats) >= num_ring1:
            break
        if cat != sme_cat and cat not in ring1_cats:
            ring1_cats.append(cat)

    ring1_nodes = []
    r1_radius = 1.0
    for i, cat in enumerate(ring1_cats):
        angle = 2 * math.pi * i / max(len(ring1_cats), 1)
        merchant = random.choice(EXPANDED_MERCHANT_NAMES.get(cat, [f"{cat} Merchant"]))
        overlap = int(affinities.get(cat, random.uniform(0.2, 0.4)) * 100)
        n_info = {
            "id": merchant,
            "label": merchant,
            "category": cat,
            "level": 1,
            "size": 28 + (overlap / 5),
            "color": CATEGORY_COLORS.get(cat, "#64748b"),
            "overlap": overlap,
            "x": r1_radius * math.cos(angle),
            "y": r1_radius * math.sin(angle),
        }
        nodes.append(n_info)
        ring1_nodes.append(n_info)

    # Ring 2: Secondary Connected Ecosystem (Level 2)
    num_ring2 = total_nodes - len(nodes)
    if num_ring2 > 0:
        r2_radius = 1.9
        for j in range(num_ring2):
            angle = (2 * math.pi * j / num_ring2) + 0.15
            cat = all_cats[j % len(all_cats)]
            merchant = random.choice(EXPANDED_MERCHANT_NAMES.get(cat, [f"Local {cat}"]))
            if any(n["id"] == merchant for n in nodes):
                merchant = f"{merchant} ({j+1})"

            parent_r1 = ring1_nodes[j % len(ring1_nodes)]
            overlap = max(10, parent_r1["overlap"] - random.randint(10, 25))

            nodes.append({
                "id": merchant,
                "label": merchant,
                "category": cat,
                "level": 2,
                "size": 18 + (overlap / 6),
                "color": CATEGORY_COLORS.get(cat, "#94a3b8"),
                "overlap": overlap,
                "parent_id": parent_r1["id"],
                "x": r2_radius * math.cos(angle) + random.uniform(-0.15, 0.15),
                "y": r2_radius * math.sin(angle) + random.uniform(-0.15, 0.15),
            })

    # Construct Plotly Edges
    edge_traces = []
    for n in nodes:
        if n["level"] == 1:
            edge_traces.append(go.Scatter(
                x=[0.0, n["x"]], y=[0.0, n["y"]],
                mode="lines",
                line=dict(width=1.5 + (n["overlap"] / 15), color="rgba(30, 58, 138, 0.35)"),
                hoverinfo="text",
                text=f"{sme_short} ↔ {n['label']}: {n['overlap']}% Customer Overlap",
                showlegend=False,
            ))
        elif n["level"] == 2:
            parent = next((p for p in nodes if p["id"] == n.get("parent_id")), None)
            if parent:
                edge_traces.append(go.Scatter(
                    x=[parent["x"], n["x"]], y=[parent["y"], n["y"]],
                    mode="lines",
                    line=dict(width=1.0, color="rgba(148, 163, 184, 0.25)"),
                    hoverinfo="skip",
                    showlegend=False,
                ))

    # Node Scatter Plots
    node_traces = []

    # Center Node Trace
    node_traces.append(go.Scatter(
        x=[0.0], y=[0.0],
        mode="markers+text",
        marker=dict(size=nodes[0]["size"], color=nodes[0]["color"], line=dict(width=3, color="white")),
        text=[sme_short],
        textposition="top center",
        textfont=dict(size=12, color="#1e293b", family="Arial Black"),
        hoverinfo="text",
        hovertext=f"<b>{sme_short}</b> (Your Business)<br>Category: {sme_cat}",
        name="Your SME",
        showlegend=True,
    ))

    # Group outer nodes by category for clean legend
    cat_groups = {}
    for n in nodes[1:]:
        c = n["category"]
        cat_groups.setdefault(c, []).append(n)

    for cat, cat_nodes in cat_groups.items():
        node_traces.append(go.Scatter(
            x=[n["x"] for n in cat_nodes],
            y=[n["y"] for n in cat_nodes],
            mode="markers+text",
            marker=dict(
                size=[n["size"] for n in cat_nodes],
                color=[n["color"] for n in cat_nodes],
                line=dict(width=1.5, color="white")
            ),
            text=[n["label"] if n["level"] == 1 else "" for n in cat_nodes],
            textposition="bottom center",
            textfont=dict(size=10, color="#334155"),
            hoverinfo="text",
            hovertext=[
                f"<b>{n['label']}</b><br>Category: {n['category']}<br>Customer Overlap: {n['overlap']}%"
                for n in cat_nodes
            ],
            name=cat,
            showlegend=True,
        ))

    fig = go.Figure(data=edge_traces + node_traces)
    fig.update_layout(
        title=dict(
            text=f"<b>Visa Aggregated Spend Network — {sme_short}</b> ({len(nodes)} Local Ecosystem Merchants)",
            x=0.02, y=0.96, font=dict(size=16)
        ),
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5),
        xaxis=dict(visible=False, range=[-2.5, 2.5]),
        yaxis=dict(visible=False, range=[-2.5, 2.5], scaleanchor="x", scaleratio=1),
        margin=dict(l=20, r=20, t=50, b=50),
        height=580,
        plot_bgcolor="#f8fafc",
    )
    return fig


def run_lookalike(sme_name: str, top_n: int, total_nodes: int):
    category = SME_PROFILES[sme_name]["category"]
    affinities = CATEGORY_AFFINITY.get(category, {})
    if not affinities:
        empty_fig = go.Figure()
        empty_fig.update_layout(title="No affinity data available for this category yet.")
        return (
            pd.DataFrame({"Message": ["No affinity data available for this category yet."]}),
            "",
            empty_fig,
        )

    sorted_aff = sorted(affinities.items(), key=lambda x: -x[1])[: int(top_n)]

    rows = []
    pitches = []
    for cat, overlap in sorted_aff:
        merchant = random.choice(EXPANDED_MERCHANT_NAMES.get(cat, ["Local Partner"]))
        rows.append({
            "Partner Category": cat,
            "Suggested Merchant": merchant,
            "Customer Overlap": f"{int(overlap * 100)}%",
        })
        sme_short = sme_name.split(" (")[0]
        pitches.append(
            f"**{sme_short} × {merchant}** ({int(overlap*100)}% customer overlap)\n"
            f"→ Idea: Bundle a joint offer — e.g. show a receipt from {merchant} at "
            f"{sme_short} (or vice versa) this month for a shared discount. "
            "Cross-promote via each other's WhatsApp broadcast list / Instagram stories."
        )

    table = pd.DataFrame(rows)
    pitch_text = "\n\n".join(pitches)
    graph_fig = build_large_network_graph(sme_name, total_nodes=int(total_nodes))
    return table, pitch_text, graph_fig


def on_partner_select(
    evt: gr.SelectData, table_df: pd.DataFrame, sme_name: str
):
    if table_df is None or len(table_df) == 0:
        return "", "", "", ""

    row_idx = evt.index[0]
    row = table_df.iloc[row_idx]
    overlap_pct = int(str(row["Customer Overlap"]).strip("%"))

    offer = generate_joint_offer(
        sme_name=sme_name,
        partner_name=row["Suggested Merchant"],
        partner_category=row["Partner Category"],
        overlap_pct=overlap_pct,
    )

    insight_md = (
        f"### 🤝 Joint Offer — {sme_name.split(' (')[0]} ×"
        f" {row['Suggested Merchant']}\n{offer['insight']}"
    )
    email_full = f"Subject: {offer['email_subject']}\n\n{offer['email_body']}"
    return insight_md, offer["whatsapp"], offer["sms"], email_full


def load_raw_transactions(sme_name: str, search_query: str):
    df = generate_transactions(sme_name)

    if search_query and search_query.strip():
        q = search_query.strip().lower()
        df = df[
            df["customer_id"].str.lower().str.contains(q)
            | df["transaction_id"].str.lower().str.contains(q)
            | df["payment_method"].str.lower().str.contains(q)
        ]

    total_txns = len(df)
    total_revenue = df["amount_aed"].sum() if total_txns > 0 else 0
    avg_ticket = df["amount_aed"].mean() if total_txns > 0 else 0
    unique_custs = df["customer_id"].nunique() if total_txns > 0 else 0

    stats_html = f"""
    <div style="display: flex; gap: 15px; margin-bottom: 10px;">
        <div style="flex: 1; padding: 12px; background: #f8fafc; border-radius: 8px; border: 1px solid #e2e8f0;">
            <div style="font-size: 0.8em; color: #64748b; font-weight: 600;">TOTAL TRANSACTIONS</div>
            <div style="font-size: 1.4em; font-weight: bold; color: #1e293b;">{total_txns:,}</div>
        </div>
        <div style="flex: 1; padding: 12px; background: #f8fafc; border-radius: 8px; border: 1px solid #e2e8f0;">
            <div style="font-size: 0.8em; color: #64748b; font-weight: 600;">TOTAL REVENUE</div>
            <div style="font-size: 1.4em; font-weight: bold; color: #059669;">AED {total_revenue:,.2f}</div>
        </div>
        <div style="flex: 1; padding: 12px; background: #f8fafc; border-radius: 8px; border: 1px solid #e2e8f0;">
            <div style="font-size: 0.8em; color: #64748b; font-weight: 600;">AVG TICKET SIZE</div>
            <div style="font-size: 1.4em; font-weight: bold; color: #2563eb;">AED {avg_ticket:.2f}</div>
        </div>
        <div style="flex: 1; padding: 12px; background: #f8fafc; border-radius: 8px; border: 1px solid #e2e8f0;">
            <div style="font-size: 0.8em; color: #64748b; font-weight: 600;">UNIQUE CUSTOMERS</div>
            <div style="font-size: 1.4em; font-weight: bold; color: #d97706;">{unique_custs}</div>
        </div>
    </div>
    """

    if total_txns > 0:
        fig = px.histogram(
            df,
            x="amount_aed",
            nbins=25,
            title=f"Transaction Amount Distribution — {sme_name.split(' (')[0]}",
            labels={"amount_aed": "Transaction Amount (AED)"},
            color_discrete_sequence=["#6366f1"],
        )
        fig.update_layout(
            margin=dict(l=20, r=20, t=40, b=20),
            height=280,
            plot_bgcolor="white",
            yaxis_title="Count",
        )
    else:
        fig = go.Figure()

    return df, stats_html, fig


# ----------------------------------------------------------------------
# 3. AI ROI ENGINE CALCULATION & VISUALIZATION LOGIC
# ----------------------------------------------------------------------

def calculate_joint_roi(sme_name: str, partner_category: str, discount_pct: float, target_reach: int):
    """
    Predicts double-sided campaign ROI for both the initiating SME and Partner Merchant.
    """
    sme_profile = SME_PROFILES[sme_name]
    sme_cat = sme_profile["category"]
    sme_ticket = sme_profile["avg_ticket"]

    cat_avg_tickets = {
        "Food & Beverage": 45, "Fitness & Wellness": 220, "Personal Care": 175,
        "Retail - Fashion": 210, "Grocery": 120, "Entertainment": 150,
        "Specialty Coffee": 35, "Luxury Apparel": 650, "Spa & Wellness": 280
    }
    partner_ticket = cat_avg_tickets.get(partner_category, 150)

    affinity = CATEGORY_AFFINITY.get(sme_cat, {}).get(partner_category, 0.25)

    # 1. Initiating SME ROI Prediction
    base_conv = affinity * 0.12
    discount_boost = 1 + (discount_pct / 100) * 1.5
    sme_conv_rate = min(base_conv * discount_boost, 0.35)

    sme_new_custs = int(target_reach * sme_conv_rate)
    sme_gross_rev = sme_new_custs * sme_ticket
    sme_discount_cost = sme_new_custs * (sme_ticket * (discount_pct / 100))
    sme_net_profit = sme_gross_rev - sme_discount_cost
    sme_roi = (sme_net_profit / sme_discount_cost * 100) if sme_discount_cost > 0 else 0

    # 2. Partner Merchant ROI Prediction (Reciprocal back-flow)
    reciprocal_rate = 0.65
    partner_new_custs = int(sme_new_custs * reciprocal_rate)
    partner_gross_rev = partner_new_custs * partner_ticket
    partner_discount_cost = partner_new_custs * (partner_ticket * (discount_pct / 100))
    partner_net_profit = partner_gross_rev - partner_discount_cost
    partner_roi = (partner_net_profit / partner_discount_cost * 100) if partner_discount_cost > 0 else 0

    # Summary Metrics Table
    summary_data = {
        "Metric": [
            "Average Ticket Size",
            "Est. Converted Customers",
            "Gross Revenue Generated",
            "Total Discount Cost",
            "Net Profit",
            "Predicted Campaign ROI"
        ],
        f"Your SME ({sme_name.split(' (')[0]})": [
            f"AED {sme_ticket:.2f}",
            f"{sme_new_custs:,}",
            f"AED {sme_gross_rev:,.2f}",
            f"AED {sme_discount_cost:,.2f}",
            f"AED {sme_net_profit:,.2f}",
            f"{sme_roi:,.1f}%"
        ],
        f"Partner ({partner_category})": [
            f"AED {partner_ticket:.2f}",
            f"{partner_new_custs:,}",
            f"AED {partner_gross_rev:,.2f}",
            f"AED {partner_discount_cost:,.2f}",
            f"AED {partner_net_profit:,.2f}",
            f"{partner_roi:,.1f}%"
        ]
    }
    summary_df = pd.DataFrame(summary_data)

    # HTML Header KPI Cards
    kpi_html = f"""
    <div style="display: flex; gap: 15px; margin-bottom: 15px;">
        <div style="flex: 1; padding: 14px; background: #e0f2fe; border-radius: 8px; border: 1px solid #bae6fd;">
            <div style="font-size: 0.75em; color: #0369a1; font-weight: 700; text-transform: uppercase;">PREDICTED CONVERSION</div>
            <div style="font-size: 1.5em; font-weight: bold; color: #0284c7;">{sme_conv_rate * 100:.1f}%</div>
            <div style="font-size: 0.8em; color: #0369a1;">Based on {int(affinity*100)}% Visa category affinity</div>
        </div>
        <div style="flex: 1; padding: 14px; background: #dcfce7; border-radius: 8px; border: 1px solid #bbf7d0;">
            <div style="font-size: 0.75em; color: #15803d; font-weight: 700; text-transform: uppercase;">COMBINED NET PROFIT</div>
            <div style="font-size: 1.5em; font-weight: bold; color: #16a34a;">AED {sme_net_profit + partner_net_profit:,.2f}</div>
            <div style="font-size: 0.8em; color: #15803d;">Shared value created across ecosystem</div>
        </div>
        <div style="flex: 1; padding: 14px; background: #fef3c7; border-radius: 8px; border: 1px solid #fde68a;">
            <div style="font-size: 0.75em; color: #b45309; font-weight: 700; text-transform: uppercase;">YOUR SME NET ROI</div>
            <div style="font-size: 1.5em; font-weight: bold; color: #d97706;">{sme_roi:,.1f}%</div>
            <div style="font-size: 0.8em; color: #b45309;">Incremental revenue vs discount cost</div>
        </div>
        <div style="flex: 1; padding: 14px; background: #f3e8ff; border-radius: 8px; border: 1px solid #e9d5ff;">
            <div style="font-size: 0.75em; color: #6b21a8; font-weight: 700; text-transform: uppercase;">PARTNER NET ROI</div>
            <div style="font-size: 1.5em; font-weight: bold; color: #9333ea;">{partner_roi:,.1f}%</div>
            <div style="font-size: 0.8em; color: #6b21a8;">Back-flow customer retention</div>
        </div>
    </div>
    """

    # Financial Breakdown Chart
    chart_df = pd.DataFrame([
        {"Merchant": f"Your SME ({sme_name.split(' (')[0]})", "Type": "Gross Revenue", "Amount": sme_gross_rev},
        {"Merchant": f"Your SME ({sme_name.split(' (')[0]})", "Type": "Discount Cost", "Amount": sme_discount_cost},
        {"Merchant": f"Your SME ({sme_name.split(' (')[0]})", "Type": "Net Profit", "Amount": sme_net_profit},
        {"Merchant": f"Partner ({partner_category})", "Type": "Gross Revenue", "Amount": partner_gross_rev},
        {"Merchant": f"Partner ({partner_category})", "Type": "Discount Cost", "Amount": partner_discount_cost},
        {"Merchant": f"Partner ({partner_category})", "Type": "Net Profit", "Amount": partner_net_profit},
    ])

    fig = px.bar(
        chart_df,
        x="Merchant",
        y="Amount",
        color="Type",
        barmode="group",
        title="Predictive Campaign Economics: Revenue vs Discount Cost vs Net Profit",
        color_discrete_map={"Gross Revenue": "#3b82f6", "Discount Cost": "#ef4444", "Net Profit": "#10b981"},
        labels={"Amount": "Amount (AED)"}
    )
    fig.update_layout(
        height=320,
        margin=dict(l=20, r=20, t=40, b=20),
        plot_bgcolor="white"
    )

    return kpi_html, summary_df, fig


def update_partner_categories(sme_name: str):
    sme_cat = SME_PROFILES[sme_name]["category"]
    affinities = CATEGORY_AFFINITY.get(sme_cat, {})
    cats = list(affinities.keys()) if affinities else list(EXPANDED_MERCHANT_NAMES.keys())
    return gr.Dropdown(choices=cats, value=cats[0] if cats else None)


# ----------------------------------------------------------------------
# 3b. LOCATION ANALYTICS
#     Where is demand strongest, and where does this SME's category face
#     the least competition? Built on a synthetic Dubai-area demand /
#     competitor-density model — swap for real Visa merchant-density +
#     footfall data in production.
# ----------------------------------------------------------------------

DUBAI_AREAS = {
    "Jumeirah": (25.2285, 55.2593),
    "Al Barsha": (25.1121, 55.2044),
    "Dubai Marina": (25.0805, 55.1403),
    "JLT": (25.0693, 55.1465),
    "Business Bay": (25.1877, 55.2633),
    "Downtown Dubai": (25.1972, 55.2744),
    "Deira": (25.2697, 55.3095),
    "Al Quoz": (25.1367, 55.2268),
    "Mirdif": (25.2159, 55.4235),
    "Arabian Ranches": (25.0537, 55.2708),
}

SME_AREA = {
    "Café (Jumeirah)": "Jumeirah",
    "Boutique Gym (Al Barsha)": "Al Barsha",
    "Hair & Beauty Salon (Marina)": "Dubai Marina",
    "Independent Retail Store (JLT)": "JLT",
}


def _area_category_stats(area: str, category: str) -> dict:
    """Deterministic synthetic demand/competition stats for an
    (area, category) pair — stands in for aggregated Visa merchant-density
    and spend-demand data in production."""
    demand_rnd = random.Random(f"demand::{area}::{category}")
    comp_rnd = random.Random(f"competitors::{area}::{category}")
    demand_index = round(demand_rnd.uniform(35, 95), 1)     # 0-100 relative demand
    competitor_count = comp_rnd.randint(2, 26)               # merchants already in that category/area
    opportunity_score = round(demand_index - competitor_count * 2.2, 1)
    return {
        "demand_index": demand_index,
        "competitor_count": competitor_count,
        "opportunity_score": opportunity_score,
    }


def compute_location_analytics(sme_name: str):
    home_area = SME_AREA.get(sme_name, "Jumeirah")
    category = SME_PROFILES[sme_name]["category"]
    home_lat, home_lon = DUBAI_AREAS[home_area]

    rows = []
    for area, (lat, lon) in DUBAI_AREAS.items():
        stats = _area_category_stats(area, category)
        rows.append({
            "Area": area,
            "lat": lat,
            "lon": lon,
            "is_home": area == home_area,
            **stats,
        })
    area_df = pd.DataFrame(rows)

    # Map: bubble size = demand, color = opportunity score
    map_fig = px.scatter_map(
        area_df,
        lat="lat", lon="lon",
        size="demand_index",
        color="opportunity_score",
        color_continuous_scale="RdYlGn",
        hover_name="Area",
        hover_data={
            "lat": False, "lon": False, "is_home": False,
            "demand_index": True, "competitor_count": True, "opportunity_score": True,
        },
        size_max=32,
        zoom=10.2,
        center={"lat": 25.16, "lon": 55.22},
        title=f"Location Demand & Competition Map — {sme_name.split(' (')[0]} ({category})",
    )
    map_fig.update_layout(
        map_style="open-street-map",
        height=460,
        margin=dict(l=10, r=10, t=45, b=10),
    )
    # Mark the SME's own location distinctly
    map_fig.add_trace(go.Scattermap(
        lat=[home_lat], lon=[home_lon],
        mode="markers+text",
        marker=dict(size=18, color="#1d4ed8"),
        text=["📍 You are here"],
        textposition="top right",
        hoverinfo="text",
        hovertext=f"{sme_name} (current location)",
        name="Your SME",
    ))

    # Expansion opportunity ranking (exclude home area)
    opp_df = area_df[~area_df["is_home"]].sort_values("opportunity_score", ascending=False)
    opp_table = opp_df[["Area", "demand_index", "competitor_count", "opportunity_score"]].rename(
        columns={
            "demand_index": "Demand Index (0-100)",
            "competitor_count": "Existing Competitors",
            "opportunity_score": "Opportunity Score",
        }
    ).reset_index(drop=True)

    top = opp_df.iloc[0]
    home_stats = area_df[area_df["is_home"]].iloc[0]
    narrative = (
        f"### 📍 Location Insight — {sme_name.split(' (')[0]}\n"
        f"Your current area (**{home_area}**) has a demand index of "
        f"**{home_stats['demand_index']}/100** with **{int(home_stats['competitor_count'])}** "
        f"existing **{category}** competitors nearby (opportunity score "
        f"**{home_stats['opportunity_score']}**).\n\n"
        f"The strongest **expansion opportunity** is **{top['Area']}** — demand index "
        f"**{top['demand_index']}/100** with only **{int(top['competitor_count'])}** existing "
        f"competitors (opportunity score **{top['opportunity_score']}**), suggesting under-served "
        f"demand for a second location or delivery/partnership reach into that area."
    )

    return map_fig, opp_table, narrative


# ----------------------------------------------------------------------
# 3c. MERCHANT INTELLIGENCE BENCHMARKING
#     How does this SME compare to category peers on core performance
#     metrics? Benchmarks below are an illustrative synthetic baseline —
#     swap for real aggregated Visa merchant-category benchmark data in
#     production.
# ----------------------------------------------------------------------

CATEGORY_BENCHMARKS = {
    "Food & Beverage": {
        "avg_ticket_aed": (42, 9),
        "retention_rate_pct": (52, 9),
        "visit_frequency": (5.0, 1.4),
        "repeat_customer_rate_pct": (58, 10),
    },
    "Fitness & Wellness": {
        "avg_ticket_aed": (230, 45),
        "retention_rate_pct": (60, 10),
        "visit_frequency": (6.2, 1.6),
        "repeat_customer_rate_pct": (65, 9),
    },
    "Personal Care": {
        "avg_ticket_aed": (170, 35),
        "retention_rate_pct": (55, 9),
        "visit_frequency": (4.6, 1.3),
        "repeat_customer_rate_pct": (60, 10),
    },
    "Retail - Fashion": {
        "avg_ticket_aed": (205, 40),
        "retention_rate_pct": (48, 9),
        "visit_frequency": (3.4, 1.1),
        "repeat_customer_rate_pct": (50, 10),
    },
}


def _ordinal(n: int) -> str:
    if 10 <= n % 100 <= 20:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"


def _normal_percentile(value: float, mean: float, std: float) -> float:
    """Percentile (0-100) of `value` under a Normal(mean, std) — avoids a
    scipy dependency by using math.erf directly."""
    if std <= 0:
        return 50.0
    z = (value - mean) / (std * math.sqrt(2))
    return round(50 * (1 + math.erf(z)), 1)


def compute_merchant_benchmarking(sme_name: str):
    category = SME_PROFILES[sme_name]["category"]
    benchmarks = CATEGORY_BENCHMARKS.get(category)
    if not benchmarks:
        empty_fig = go.Figure()
        empty_fig.update_layout(title="No benchmark data available for this category yet.")
        return empty_fig, pd.DataFrame({"Message": ["No benchmark data for this category."]}), ""

    df = generate_transactions(sme_name)
    rfm = compute_rfm(df)
    total = len(rfm)

    avg_ticket = df["amount_aed"].mean()
    retention_rate = 100 * (1 - (rfm["segment"] == "Win-Back (Lapsing)").sum() / total)
    visit_frequency = rfm["frequency"].mean()
    repeat_rate = 100 * (rfm["frequency"] >= 2).sum() / total

    actual = {
        "avg_ticket_aed": avg_ticket,
        "retention_rate_pct": retention_rate,
        "visit_frequency": visit_frequency,
        "repeat_customer_rate_pct": repeat_rate,
    }

    labels = {
        "avg_ticket_aed": "Avg Ticket Size (AED)",
        "retention_rate_pct": "Retention Rate (%)",
        "visit_frequency": "Visit Frequency (visits/customer)",
        "repeat_customer_rate_pct": "Repeat Customer Rate (%)",
    }

    rows = []
    percentiles = {}
    for key, (mean, std) in benchmarks.items():
        pct = _normal_percentile(actual[key], mean, std)
        percentiles[key] = pct
        rows.append({
            "Metric": labels[key],
            "Your SME": round(actual[key], 1),
            f"{category} Category Average": mean,
            "Percentile vs. Peers": _ordinal(int(round(pct))),
        })
    bench_table = pd.DataFrame(rows)

    # Radar chart: percentile vs. peers on each metric (50th = category average)
    radar_labels = [labels[k] for k in benchmarks.keys()]
    radar_values = [percentiles[k] for k in benchmarks.keys()]
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=radar_values + [radar_values[0]],
        theta=radar_labels + [radar_labels[0]],
        fill="toself",
        name="Your SME (percentile)",
        line=dict(color="#1d4ed8"),
    ))
    fig.add_trace(go.Scatterpolar(
        r=[50] * (len(radar_labels) + 1),
        theta=radar_labels + [radar_labels[0]],
        mode="lines",
        line=dict(color="#94a3b8", dash="dash"),
        name=f"{category} Category Average (50th pct.)",
    ))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
        title=f"Benchmark Percentile vs. {category} Peers — {sme_name.split(' (')[0]}",
        height=440,
        margin=dict(l=40, r=40, t=60, b=40),
        showlegend=True,
    )

    strongest = max(percentiles, key=percentiles.get)
    weakest = min(percentiles, key=percentiles.get)
    narrative = (
        f"### 📊 Benchmark Insight — {sme_name.split(' (')[0]}\n"
        f"Strongest vs. peers: **{labels[strongest]}** at the "
        f"**{_ordinal(int(round(percentiles[strongest])))} percentile**. Weakest: **{labels[weakest]}** "
        f"at the **{_ordinal(int(round(percentiles[weakest])))} percentile** — this is the metric most "
        "worth targeting with a campaign from the AI Marketing Co-Pilot."
    )

    return fig, bench_table, narrative


# ----------------------------------------------------------------------
# 3d. CASHFLOW FORECASTING
#     Projects weekly revenue and cumulative cash balance forward, with
#     and without the uplift from running campaigns — ties the ROI Engine
#     back into a runway/cashflow view an SME owner actually cares about.
# ----------------------------------------------------------------------

def compute_cashflow_forecast(
    sme_name: str,
    weeks_ahead: int,
    campaign_uplift_pct: float,
    starting_cash_aed: float,
    weekly_fixed_costs_aed: float,
):
    df = generate_transactions(sme_name)
    df["date_dt"] = pd.to_datetime(df["date"])
    df["week"] = df["date_dt"].dt.to_period("W").apply(lambda p: p.start_time)

    weekly = df.groupby("week")["amount_aed"].sum().sort_index()
    weekly = weekly[weekly.index <= pd.Timestamp(datetime.today())]  # drop any partial future bucket

    if len(weekly) < 3:
        empty_fig = go.Figure()
        empty_fig.update_layout(title="Not enough transaction history to forecast yet.")
        return empty_fig, pd.DataFrame({"Message": ["Not enough transaction history."]}), ""

    weeks_ahead = int(weeks_ahead)
    x = np.arange(len(weekly))
    y = weekly.values.astype(float)

    # Simple linear trend + residual noise level, extrapolated forward
    slope, intercept = np.polyfit(x, y, 1)
    residual_std = float(np.std(y - (slope * x + intercept)))

    future_x = np.arange(len(weekly), len(weekly) + weeks_ahead)
    baseline_forecast = np.clip(slope * future_x + intercept, a_min=0, a_max=None)
    campaign_forecast = baseline_forecast * (1 + campaign_uplift_pct / 100)

    future_dates = [weekly.index[-1] + timedelta(weeks=i + 1) for i in range(weeks_ahead)]

    # Cumulative cash balance: starting cash + cumulative (revenue - fixed costs)
    baseline_net = baseline_forecast - weekly_fixed_costs_aed
    campaign_net = campaign_forecast - weekly_fixed_costs_aed
    baseline_balance = starting_cash_aed + np.cumsum(baseline_net)
    campaign_balance = starting_cash_aed + np.cumsum(campaign_net)

    # ---- Chart: historical + both forecasts ----
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=list(weekly.index), y=list(y),
        mode="lines+markers", name="Actual Weekly Revenue",
        line=dict(color="#1d4ed8"),
    ))
    fig.add_trace(go.Scatter(
        x=future_dates, y=list(baseline_forecast),
        mode="lines+markers", name="Forecast (No Campaigns)",
        line=dict(color="#94a3b8", dash="dash"),
    ))
    fig.add_trace(go.Scatter(
        x=future_dates, y=list(campaign_forecast),
        mode="lines+markers", name=f"Forecast (+{campaign_uplift_pct:.0f}% Campaign Uplift)",
        line=dict(color="#16a34a", dash="dash"),
    ))
    fig.update_layout(
        title=f"Weekly Revenue Forecast — {sme_name.split(' (')[0]}",
        xaxis_title="Week", yaxis_title="Revenue (AED)",
        height=380, margin=dict(l=20, r=20, t=45, b=20), plot_bgcolor="white",
    )

    # ---- Cumulative cash balance chart data folded into summary table ----
    summary_rows = [
        {"Metric": "Weeks Forecasted", "Baseline (No Campaigns)": weeks_ahead, "With Campaign Uplift": weeks_ahead},
        {
            "Metric": "Total Projected Revenue",
            "Baseline (No Campaigns)": f"AED {baseline_forecast.sum():,.0f}",
            "With Campaign Uplift": f"AED {campaign_forecast.sum():,.0f}",
        },
        {
            "Metric": "Total Fixed Costs",
            "Baseline (No Campaigns)": f"AED {weekly_fixed_costs_aed * weeks_ahead:,.0f}",
            "With Campaign Uplift": f"AED {weekly_fixed_costs_aed * weeks_ahead:,.0f}",
        },
        {
            "Metric": f"Cash Balance After {weeks_ahead} Weeks",
            "Baseline (No Campaigns)": f"AED {baseline_balance[-1]:,.0f}",
            "With Campaign Uplift": f"AED {campaign_balance[-1]:,.0f}",
        },
        {
            "Metric": "Incremental Cash from Campaigns",
            "Baseline (No Campaigns)": "—",
            "With Campaign Uplift": f"AED {campaign_balance[-1] - baseline_balance[-1]:,.0f}",
        },
    ]
    summary_table = pd.DataFrame(summary_rows)

    risk_note = ""
    if (baseline_balance < 0).any():
        weeks_to_negative = int(np.argmax(baseline_balance < 0)) + 1
        risk_note = (
            f"\n\n⚠️ **Cash risk:** at current trend and costs, the baseline (no "
            f"campaign) cash balance is projected to go negative around **week "
            f"{weeks_to_negative}**."
        )
        if (campaign_balance >= 0).all():
            risk_note += " Running the modeled campaigns keeps the balance positive across the full forecast window."

    narrative = (
        f"### 💰 Cashflow Insight — {sme_name.split(' (')[0]}\n"
        f"At the current revenue trend (~AED {slope:,.0f}/week change) and "
        f"AED {weekly_fixed_costs_aed:,.0f}/week fixed costs, projected cash balance after "
        f"{weeks_ahead} weeks is **AED {baseline_balance[-1]:,.0f}** without campaigns vs. "
        f"**AED {campaign_balance[-1]:,.0f}** with a {campaign_uplift_pct:.0f}% campaign uplift "
        f"applied — an incremental **AED {campaign_balance[-1] - baseline_balance[-1]:,.0f}** "
        f"in cash generated.{risk_note}"
    )

    return fig, summary_table, narrative


# ----------------------------------------------------------------------
# 4. GRADIO UI DEFINITION
# ----------------------------------------------------------------------

with gr.Blocks(title="SME Growth Co-Pilot — Visa Data+AI Hackathon") as demo:
    gr.Markdown(
        """
        # 🚀 SME Growth Co-Pilot
        **Visa Data + AI Hackathon (UAE) — SME Growth: Marketing + AI**

        Prototype combining four core modules:
        1. **AI Marketing Co-Pilot** — manage cohort contact lists, turn transaction data into
           customer segments, generate ready-to-send campaigns, and email them out — all in one place.
        2. **Look-Alike Customer Finder & Network Graph** — multi-tiered merchant cross-spend
           ecosystem visualization and joint-offer drafting.
        3. **AI ROI Engine** — simulates dual-merchant joint promotions with predictive ROI, plus
           forward cashflow forecasting.
        4. **Strategy & Expansion** — location demand/competition analytics and merchant
           benchmarking against category peers.

        """
    )

    with gr.Tab("1️⃣ AI Marketing Co-Pilot"):
        with gr.Row():
            sme_dd1 = gr.Dropdown(
                list(SME_PROFILES.keys()),
                value=list(SME_PROFILES.keys())[0],
                label="Choose SME",
            )
            seg_dd = gr.Dropdown(
                ["Win-Back (Lapsing)", "Champions", "New Customers", "Regulars"],
                value="Win-Back (Lapsing)",
                label="Focus Segment (Cohort)",
            )

        with gr.Accordion("📋 Manage Cohort Contact List (optional)", open=False):
            gr.Markdown(
                """
                Paste your real customer contacts for the **SME + Cohort selected above**,
                **one person per line**:

                ```
                Sara Ahmed, sara.ahmed@email.com
                Omar Khalid, omar.k@email.com
                ```

                Once saved, generating and sending a campaign for this cohort will use **this
                list** instead of the app's synthetic demo directory. Leave it empty to keep
                using the synthetic demo data.
                """
            )
            contacts_text = gr.Textbox(
                label="Contacts for this SME + Cohort (one per line: Name, email@example.com)",
                lines=6,
                placeholder="Sara Ahmed, sara.ahmed@email.com\nOmar Khalid, omar.k@email.com",
            )
            with gr.Row():
                save_contacts_btn = gr.Button("💾 Save List for This Cohort", variant="primary")
                clear_contacts_btn = gr.Button("🗑️ Clear List for This Cohort")
                preview_contacts_btn = gr.Button("👀 Refresh Preview")
            contacts_status = gr.Markdown()
            cohort_contacts_table = gr.Dataframe(label="Currently Saved Contacts for This SME + Cohort")

            save_contacts_btn.click(
                save_cohort_contacts,
                inputs=[sme_dd1, seg_dd, contacts_text],
                outputs=[contacts_status],
            ).then(
                preview_cohort_contacts,
                inputs=[sme_dd1, seg_dd],
                outputs=[cohort_contacts_table],
            )

            clear_contacts_btn.click(
                clear_cohort_contacts,
                inputs=[sme_dd1, seg_dd],
                outputs=[contacts_status],
            ).then(
                preview_cohort_contacts,
                inputs=[sme_dd1, seg_dd],
                outputs=[cohort_contacts_table],
            )

            preview_contacts_btn.click(
                preview_cohort_contacts,
                inputs=[sme_dd1, seg_dd],
                outputs=[cohort_contacts_table],
            )

            sme_dd1.change(
                preview_cohort_contacts,
                inputs=[sme_dd1, seg_dd],
                outputs=[cohort_contacts_table],
            )
            seg_dd.change(
                preview_cohort_contacts,
                inputs=[sme_dd1, seg_dd],
                outputs=[cohort_contacts_table],
            )

        run_btn1 = gr.Button("🔍 Analyze & Generate Campaign", variant="primary")

        seg_table = gr.Dataframe(label="Customer Segments (RFM Analysis)")
        insight_md = gr.Markdown()

        with gr.Row():
            whatsapp_box = gr.Textbox(label="📱 WhatsApp Draft", lines=4)
            sms_box = gr.Textbox(label="💬 SMS Draft", lines=4)
        email_box = gr.Textbox(label="📧 Email Draft", lines=6)

        run_btn1.click(
            run_copilot,
            inputs=[sme_dd1, seg_dd],
            outputs=[seg_table, insight_md, whatsapp_box, sms_box, email_box],
        )

        with gr.Accordion("📧 Send This Campaign via Email", open=False):
            gr.Markdown(
                "Sends the **Email Draft** above to every customer currently in the "
                "selected cohort (segment). If you saved a custom name + email list "
                "for this SME + cohort above, that list is used; otherwise the "
                "synthetic demo directory is used. Uses SMTP (e.g. Gmail with an "
                "**App Password**, not your normal password). Leave **Dry Run** on to "
                "preview who would be emailed without sending anything."
            )
            preview_btn = gr.Button("👀 Preview Recipients in Cohort")
            recipients_table = gr.Dataframe(label="Recipients (name, email, RFM)")

            with gr.Row():
                smtp_host = gr.Textbox(label="SMTP Host", value="smtp.gmail.com")
                smtp_port = gr.Number(label="SMTP Port", value=465, precision=0)
                sender_email = gr.Textbox(label="Sender Email", placeholder="you@yourbusiness.com")
                app_password = gr.Textbox(label="App Password", type="password")
            with gr.Row():
                dry_run = gr.Checkbox(label="Dry Run (preview only — no emails sent)", value=True)
                max_recipients = gr.Slider(1, 50, value=5, step=1, label="Max Recipients (safety cap)")

            send_btn = gr.Button("📤 Send Campaign to Cohort", variant="primary")
            send_status = gr.Markdown()

            preview_btn.click(
                preview_segment_recipients,
                inputs=[sme_dd1, seg_dd, max_recipients],
                outputs=[recipients_table],
            )

            send_btn.click(
                send_campaign_emails,
                inputs=[
                    sme_dd1, seg_dd, email_box,
                    smtp_host, smtp_port, sender_email, app_password,
                    dry_run, max_recipients,
                ],
                outputs=[send_status],
            )

    with gr.Tab("2️⃣ Look-Alike Customer Finder & Network Graph"):
        with gr.Row():
            sme_dd2 = gr.Dropdown(
                list(SME_PROFILES.keys()),
                value=list(SME_PROFILES.keys())[0],
                label="Choose SME",
            )
            top_n = gr.Slider(
                1, 5, value=3, step=1, label="Number of Direct Partner Suggestions"
            )
            total_nodes = gr.Slider(
                10, 40, value=25, step=5, label="Network Graph Ecosystem Size (Nodes)"
            )
        run_btn2 = gr.Button("🔗 Generate Partner Network Graph", variant="primary")

        network_plot = gr.Plot(label="Ecosystem Spend Network")
        partner_table = gr.Dataframe(
            label=(
                "Top Co-Marketing Partners — click a row to generate a joint"
                " offer draft"
            )
        )
        pitch_md = gr.Markdown(label="Co-Marketing Ideas")

        gr.Markdown(
            "### 🤝 Joint Offer Drafts\n*Click a row in the table above to generate"
            " matching campaign copy.*"
        )
        joint_insight_md = gr.Markdown()
        with gr.Row():
            joint_whatsapp = gr.Textbox(label="📱 WhatsApp Draft", lines=4)
            joint_sms = gr.Textbox(label="💬 SMS Draft", lines=4)
        joint_email = gr.Textbox(label="📧 Email Draft", lines=6)

        run_btn2.click(
            run_lookalike,
            inputs=[sme_dd2, top_n, total_nodes],
            outputs=[partner_table, pitch_md, network_plot],
        )

        partner_table.select(
            on_partner_select,
            inputs=[partner_table, sme_dd2],
            outputs=[joint_insight_md, joint_whatsapp, joint_sms, joint_email],
        )

        with gr.Accordion("📧 Send This Joint Offer via Email", open=False):
            gr.Markdown(
                "Sends the **Joint Offer Email Draft** above to your own SME's "
                "customers in a chosen cohort (e.g. target your **Champions** — "
                "your most loyal customers — with the cross-promotion first). "
                "Uses a custom contact list saved in the **AI Marketing Co-Pilot** "
                "tab for this SME + cohort if one exists, otherwise the synthetic "
                "demo directory."
            )
            joint_seg_dd = gr.Dropdown(
                ["Win-Back (Lapsing)", "Champions", "New Customers", "Regulars"],
                value="Champions",
                label="Target Cohort (from your SME's customer base)",
            )
            with gr.Row():
                smtp_host2 = gr.Textbox(label="SMTP Host", value="smtp.gmail.com")
                smtp_port2 = gr.Number(label="SMTP Port", value=465, precision=0)
                sender_email2 = gr.Textbox(label="Sender Email", placeholder="you@yourbusiness.com")
                app_password2 = gr.Textbox(label="App Password", type="password")
            with gr.Row():
                dry_run2 = gr.Checkbox(label="Dry Run (preview only — no emails sent)", value=True)
                max_recipients2 = gr.Slider(1, 50, value=5, step=1, label="Max Recipients (safety cap)")

            send_btn2 = gr.Button("📤 Send Joint Offer to Cohort", variant="primary")
            send_status2 = gr.Markdown()

            send_btn2.click(
                send_campaign_emails,
                inputs=[
                    sme_dd2, joint_seg_dd, joint_email,
                    smtp_host2, smtp_port2, sender_email2, app_password2,
                    dry_run2, max_recipients2,
                ],
                outputs=[send_status2],
            )

    with gr.Tab("3️⃣ AI ROI Engine"):
        gr.Markdown(
            """
            ### 📈 AI ROI Engine
            Simulate joint promotional deals and forecast the cashflow impact of running them —
            both driven off the same transaction data.
            """
        )

        with gr.Tabs():
            with gr.Tab("💰 Joint Campaign ROI"):
                with gr.Row():
                    sme_dd4 = gr.Dropdown(
                        list(SME_PROFILES.keys()),
                        value=list(SME_PROFILES.keys())[0],
                        label="Your SME",
                    )
                    partner_cat_dd = gr.Dropdown(
                        list(CATEGORY_AFFINITY["Food & Beverage"].keys()),
                        value="Fitness & Wellness",
                        label="Partner Industry Category",
                    )
                    discount_slider = gr.Slider(
                        5, 30, value=15, step=5, label="Joint Discount Rate Offered (%)"
                    )
                    reach_slider = gr.Slider(
                        250, 5000, value=1000, step=250, label="Partner Audience Reach (Customers)"
                    )

                calc_roi_btn = gr.Button("⚡ Predict Campaign ROI & Financial Outcomes", variant="primary")

                roi_kpi_html = gr.HTML()

                with gr.Row():
                    roi_table = gr.Dataframe(label="Side-by-Side ROI & Profit Analysis", interactive=False)
                    roi_plot = gr.Plot(label="Revenue vs. Cost Visualizer")

                calc_roi_btn.click(
                    calculate_joint_roi,
                    inputs=[sme_dd4, partner_cat_dd, discount_slider, reach_slider],
                    outputs=[roi_kpi_html, roi_table, roi_plot],
                )

                sme_dd4.change(
                    update_partner_categories,
                    inputs=[sme_dd4],
                    outputs=[partner_cat_dd],
                )

            with gr.Tab("💵 Cashflow Forecasting"):
                gr.Markdown(
                    "Projects weekly revenue and cumulative cash balance forward, with and "
                    "without the uplift from running the campaigns modeled in Joint Campaign ROI."
                )
                with gr.Row():
                    cf_sme_dd = gr.Dropdown(
                        list(SME_PROFILES.keys()),
                        value=list(SME_PROFILES.keys())[0],
                        label="Your SME",
                    )
                    cf_weeks = gr.Slider(4, 16, value=8, step=1, label="Weeks to Forecast")
                    cf_uplift = gr.Slider(0, 50, value=15, step=5, label="Expected Campaign Uplift (%)")
                with gr.Row():
                    cf_starting_cash = gr.Number(label="Starting Cash Balance (AED)", value=20000)
                    cf_fixed_costs = gr.Number(label="Weekly Fixed Costs (AED)", value=3000)

                cf_btn = gr.Button("💵 Forecast Cashflow", variant="primary")

                cf_plot = gr.Plot(label="Weekly Revenue Forecast")
                cf_narrative = gr.Markdown()
                cf_table = gr.Dataframe(label="Cashflow Summary: Baseline vs. With Campaigns")

                cf_btn.click(
                    compute_cashflow_forecast,
                    inputs=[cf_sme_dd, cf_weeks, cf_uplift, cf_starting_cash, cf_fixed_costs],
                    outputs=[cf_plot, cf_table, cf_narrative],
                )

    with gr.Tab("4️⃣ Strategy & Expansion"):
        gr.Markdown(
            """
            ### 🧭 Strategy & Expansion (Growth & Market Intelligence)
            Where should this SME expand next, and how does it stack up against category peers?
            """
        )

        with gr.Tabs():
            with gr.Tab("📍 Location Analytics"):
                gr.Markdown(
                    "Where should this SME expand next? Maps category demand against "
                    "existing competitor density across Dubai areas. *Synthetic demand/"
                    "competitor model for this demo — swap for real Visa merchant-density "
                    "and footfall data in production.*"
                )
                loc_sme_dd = gr.Dropdown(
                    list(SME_PROFILES.keys()),
                    value=list(SME_PROFILES.keys())[0],
                    label="Your SME",
                )
                loc_btn = gr.Button("🗺️ Analyze Location Opportunity", variant="primary")

                loc_map = gr.Plot(label="Demand & Competition Map")
                loc_narrative = gr.Markdown()
                loc_table = gr.Dataframe(label="Expansion Opportunity Ranking (excl. current area)")

                loc_btn.click(
                    compute_location_analytics,
                    inputs=[loc_sme_dd],
                    outputs=[loc_map, loc_table, loc_narrative],
                )

            with gr.Tab("📊 Merchant Benchmarking"):
                gr.Markdown(
                    "How does this SME compare to category peers on ticket size, retention, "
                    "visit frequency, and repeat rate? *Benchmarks are an illustrative synthetic "
                    "baseline for this demo — swap for real aggregated Visa category benchmark "
                    "data in production.*"
                )
                bench_sme_dd = gr.Dropdown(
                    list(SME_PROFILES.keys()),
                    value=list(SME_PROFILES.keys())[0],
                    label="Your SME",
                )
                bench_btn = gr.Button("📊 Benchmark vs. Category Peers", variant="primary")

                bench_narrative = gr.Markdown()
                with gr.Row():
                    bench_table = gr.Dataframe(label="Metric-by-Metric Comparison")
                    bench_radar = gr.Plot(label="Percentile vs. Peers")

                bench_btn.click(
                    compute_merchant_benchmarking,
                    inputs=[bench_sme_dd],
                    outputs=[bench_radar, bench_table, bench_narrative],
                )

    # ------------------------------------------------------------------
    # Raw Transaction Data — tucked away at the bottom of the page,
    # collapsed by default, for anyone who wants to audit the underlying
    # feed without it competing for attention with the four main modules.
    # ------------------------------------------------------------------
    with gr.Accordion("🗄️ Raw Transaction Data (view underlying feed)", open=False):
        gr.Markdown(
            "### 💳 Simulated Visa Merchant Transaction Feed\nInspect the raw,"
            " anonymized transaction logs driving the RFM segmentation and spend"
            " affinity models."
        )
        with gr.Row():
            sme_dd3 = gr.Dropdown(
                list(SME_PROFILES.keys()),
                value=list(SME_PROFILES.keys())[0],
                label="Choose SME",
            )
            search_input = gr.Textbox(
                label="Search Transactions",
                placeholder="Filter by Customer ID (e.g. C0012), TXN ID, or Payment Method...",
            )
        load_txn_btn = gr.Button("🔄 Load / Refresh Data", variant="primary")

        stats_html_output = gr.HTML()
        txn_dist_plot = gr.Plot()
        txn_table = gr.Dataframe(label="Anonymized Visa Transaction Feed", interactive=False)

        load_txn_btn.click(
            load_raw_transactions,
            inputs=[sme_dd3, search_input],
            outputs=[txn_table, stats_html_output, txn_dist_plot],
        )

        sme_dd3.change(
            load_raw_transactions,
            inputs=[sme_dd3, search_input],
            outputs=[txn_table, stats_html_output, txn_dist_plot],
        )

        search_input.submit(
            load_raw_transactions,
            inputs=[sme_dd3, search_input],
            outputs=[txn_table, stats_html_output, txn_dist_plot],
        )

    gr.Markdown(
        """
        ---
        **How this maps to real Visa data (production version):**
        - Replace `generate_transactions()` with Visa's aggregated/anonymized merchant transaction feed (MCC, ticket size, timestamp).
        - Replace `CATEGORY_AFFINITY` with real cross-category spend-overlap statistics from Visa's data (privacy-preserving, aggregated — no raw cardholder data).
        - Replace `generate_copy()` template engine with an LLM call for fully personalized, multilingual campaign copy.
        - Replace `get_customer_directory()` with your real CRM / consented customer contact list (name + email, opt-in only) per cohort.
        - For production email volume, use a transactional email provider (SendGrid, SES, Postmark) instead of raw SMTP, and store credentials as environment variables/secrets rather than typing them into the UI.
        """
    )

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7895, theme=gr.themes.Soft())

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://776ff6ab8685f1d19d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
